In [0]:
import json
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG, SCHEMA = "payment_lakehouse", "pipeline"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
spark.sql(f"USE CATALOG {CATALOG}"); spark.sql(f"USE SCHEMA {SCHEMA}")

# as_of is a parameter, never current_timestamp(): whether a payment is
# PENDING or EXPIRED depends on the date you ask, so wall-clock time would
# make a rerun of an earlier period produce today's answer.
truth = json.loads(open(f"{LANDING}/_expected_totals.json").read())
AS_OF = truth["generated_at"]
print("as_of:", AS_OF)

In [0]:
REGIONS = [("IN","INR",7,0.40),("US","USD",3,0.20),("EU","EUR",5,0.15),
           ("SG","SGD",4,0.15),("AE","AED",10,0.10)]

(spark.createDataFrame(REGIONS,
    ["region","currency_std","retry_window_days","expected_volume_share"])
   .write.mode("overwrite").saveAsTable("dim_region"))

display(spark.table("dim_region"))

In [0]:
payments = spark.table("silver_payments")
invoices = spark.table("silver_invoices")

# One invoice per payment, enforced BEFORE the join. Two invoices would fan
# the payment into two fact rows and double-count it.
w = Window.partitionBy("payment_id").orderBy(F.col("event_ts").asc())
inv = (invoices
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("payment_id","invoice_id",
            F.col("amount").alias("invoice_amount"),
            F.col("event_ts").alias("invoice_ts")))

pay = payments.select("payment_id","customer_id","region","currency","amount",
                      "payment_method", F.col("event_ts").alias("payment_ts"))

# dim_region is 5 rows — broadcast rather than shuffle the fact table.
joined = (pay.join(inv, "payment_id", "left")
             .join(F.broadcast(spark.table("dim_region")), "region", "left"))

In [0]:
from pyspark.sql import Window

fct = (joined
  .withColumn("payment_dt",   F.to_date("payment_ts"))
    .withColumn("days_waiting",
      F.floor((F.unix_timestamp(F.to_timestamp(F.lit(AS_OF)))
             - F.unix_timestamp("payment_ts")) / 86400))
  .withColumn("has_invoice",  F.col("invoice_id").isNotNull())
  # tolerance, not equality — these are doubles
  .withColumn("amounts_agree",
      F.col("has_invoice") & (F.abs(F.col("invoice_amount") - F.col("amount")) < F.lit(0.005)))
  .withColumn("state",
      F.when(F.col("amounts_agree"), F.lit("MATCHED"))
       .when(F.col("has_invoice"),   F.lit("AMOUNT_MISMATCH"))
       .when(F.col("days_waiting") > F.col("retry_window_days"), F.lit("EXPIRED"))
       .otherwise(F.lit("PENDING")))
  .withColumn("amount_variance",
      F.when(F.col("has_invoice"), F.col("invoice_amount") - F.col("amount")))
  .withColumn("as_of", F.lit(AS_OF))
  .select("payment_id","customer_id","region","currency","amount","payment_method",
          "payment_ts","payment_dt","invoice_id","invoice_amount","invoice_ts",
          "amount_variance","days_waiting","retry_window_days","state","as_of"))

fct.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("fct_payment_state")
print(f"fct_payment_state: {spark.table('fct_payment_state').count():,}")

In [0]:
STATES = ["MATCHED","AMOUNT_MISMATCH","PENDING","EXPIRED"]

recon = (spark.table("fct_payment_state")
  .groupBy("payment_dt","region")
  .agg(*[F.sum(F.when(F.col("state")==s,1).otherwise(0)).alias(s.lower()) for s in STATES],
       F.count("*").alias("total_payments"),
       F.sum("amount").alias("total_amount")))

recon.write.mode("overwrite").saveAsTable("reconciliation_daily")
display(spark.table("reconciliation_daily").orderBy("payment_dt","region").limit(10))

In [0]:
counts   = {r["state"]: r["n"] for r in
            spark.table("fct_payment_state").groupBy("state").agg(F.count("*").alias("n")).collect()}
rejected = spark.table("silver_quarantine").count()
received = spark.table("fct_payment_state").count() + rejected

print(f"--- payment state as of {AS_OF[:10]} ---")
for s in STATES: print(f"  {s:<16}: {counts.get(s,0):>6,}")
print(f"  {'REJECTED':<16}: {rejected:>6,}")
print(f"  {'received':<16}: {received:>6,}")

accounted = sum(counts.get(s,0) for s in STATES) + rejected
assert accounted == received, f"CONTROL TOTAL BROKEN: {accounted} != {received}"
print("  control total balances : True")

print("\n--- vs producer answer key ---")
ok = True
for s in STATES + ["REJECTED"]:
    exp = truth.get(s,0); act = rejected if s=="REJECTED" else counts.get(s,0)
    ok &= exp == act
    print(f"  {s:<16} expected {exp:>6,}   actual {act:>6,}   {exp==act}")
print(f"\n  ALL MATCH : {ok}")